<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/K21_Konsult_52.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/parashar21/main/images/khona21-v0.png) <br>


---



[Prithwis Mukerjee](http://www.yantrajaal.com) ||
Website - [Parashar21](https://parashar21.blogspot.com)<br>

# Khona21 — Interpretive Jyotiṣa

**Khona21** is an experimental AI-assisted Jyotiṣa interpretation platform, developed as a companion to **Parashar21**, which performs the astronomical calculations underlying traditional Indian astrology.

Named after the legendary Bengali astrologer Khona, the platform combines precomputed Rashi and Navamsha charts, Vimshottari Dasha schedules and planetary transits (Gochar) with a curated corpus of Jyotiṣa literature, currently comprising works by **James Braha** and **Harihar Majumdar**.

Rather than relying on hard-coded interpretive rules, Khona21 employs **source-grounded, multipass LLM reasoning** to examine planetary configurations, yogas, cancellations, exceptions and conflicting interpretations. An initial assessment establishes the horoscope's principal characteristics, while subsequent consultations investigate specific questions using the original reference texts and relevant astrological evidence.

This notebook, **k21_KhonaPlus**, implements the consultation workflow, integrating deterministic calculations from Parashar21 with the interpretive capabilities of modern Large Language Models.

**Khona21 explores the convergence of traditional Jyotiṣa knowledge and modern artificial intelligence, combining computational precision with source-grounded reasoning to produce systematic, transparent and auditable astrological interpretations.**

In [ ]:
from datetime import datetime
import pytz
print("\033[1m"+'জন্ম-মৃত্যু-বিবাহ, তিন না জানেন বরাহ।'+"\033[0m")
print(datetime.now(pytz.timezone('Asia/Calcutta')))

জন্ম-মৃত্যু-বিবাহ, তিন না জানেন বরাহ।
2026-11-04 05:27:03.258041+05:30


#Setup, Installs, Imports

In [ ]:
!pip -q install pypandoc
!pip -q install python-docx
from IPython.display import display, Markdown


!wget -O k21core.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21core.py
!wget -O k21fileops.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21fileops.py
from k21core import *
from k21fileops import *

In [ ]:
# ------------------------------------------------
# required for multiple imports of the same module
# ------------------------------------------------
#import k21core
#import importlib

#importlib.reload(k21core)
from k21core import *

In [ ]:
# ------------------------------------------------
# Use Claude
# ------------------------------------------------
#!pip install -q anthropic

#provider = "claude"
#model = "claude-haiku-5-5"


In [ ]:
# ------------------------------------------------
# Use OpenAI
# ------------------------------------------------
provider = "openai"

#model = "gpt-5.6-luna"
#model = "gpt-6-luna"
model = "gpt-6.1-sol"
#model = "gpt-6-astra"

In [ ]:
# ------------------------------------------------
# Set Native ID
# ------------------------------------------------
Native_id = "TopGun"

###Access Control
This notebook accesses Google Drive to access files and uses the OpenAI API. Hence it can be used ONLY from the userid of the creator. Others will have to create their own access paths

In [ ]:
# ------------------------------------------------
# Authenticate wit LLM provider using API keys stored as Colab Secret
# ------------------------------------------------
client = authenticateLLM(provider=provider)

OPENAI | User: Calcutta | Email: prithwis@yantrajaal.com | ID: user-sNAljKNmBf2igtAV2kUuqhoF


In [ ]:
# ------------------------------------------------
# Locate necessary file and set up paths to the same
# ------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
#ls /content/drive/MyDrive/Prithwis/Parashar21/K21

K21_HOME = "/content/drive/MyDrive/Prithwis/Parashar21/K21"

# ------------------------------------------------
# The Corpus Files, Braha and Harihar, are available
# in GitHub Repo and can be downloaded with wget
# ------------------------------------------------
BRAHA   = f"{K21_HOME}/Corpus/BR_Curated_22.txt"
HARIHAR = f"{K21_HOME}/Corpus/HM_Clean_07.txt"
RAMAN = f"{K21_HOME}/Corpus/Raman_08.md"
YOGS = f"{K21_HOME}/Corpus/300_Astrology_Yogas.md"



Mounted at /content/drive


## Load Files

In [ ]:
#!wget -O braha.txt -q "https://raw.githubusercontent.com/prithwis/parashar21/main/Corpus Texts/Text_Extracts/BR_Curated_22.txt"
#!wget -O harihar.txt -q "https://raw.githubusercontent.com/prithwis/parashar21/main/Corpus Texts/Text_Extracts/HM_Clean_07.txt"
#!wget -O role.txt -q https://raw.githubusercontent.com/prithwis/parashar21/main/RoleFiles/khona261029A.txt

#BRAHA   = "/content/braha.txt"
#HARIHAR = "/content/harihar.txt"
#ROLE    = "/content/role.txt"

In [ ]:
#
# Following files from Parashar21 need to be uploaded here
#

import zipfile
from google.colab import files

# Upload ZIP file
uploaded = files.upload()
zipName = next(iter(uploaded))

# Extract all files into current directory
with zipfile.ZipFile(zipName, 'r') as zf:
    zf.extractall()

print("Extracted:", zipName)

CHART = "/content/Chart.txt"
NAVAMSA = "/content/Navamsa_Chart.txt"
DASHA = "/content/Dasha.txt"
GOCHAR = "/content/Gochar_Chart.txt"
GOCHAR_EVENTS = "/content/Transit.txt"

Saving Kompute_261103_235530.zip to Kompute_261103_235530.zip
Extracted: Kompute_261103_235530.zip


# LLM Access




In [ ]:
# ------------------------------------------------
# Define the ROLE
# ------------------------------------------------
ROLE = f"{K21_HOME}/Roles/khona261029A.txt"
khona21 = readText(ROLE)

/content/drive/MyDrive/Prithwis/Parashar21/K21/Roles/khona261029A.txt | 6,774 characters


In [ ]:
# ------------------------------------------------
# Set the Context
# ------------------------------------------------
#BaseCONTEXT = buildContext(BRAHA, HARIHAR, CHART, NAVAMSA, DASHA)
'''
BaseCONTEXT = getContext(
    BRAHA=BRAHA,
    HARIHAR=HARIHAR,
    CHART=CHART,
    NAVAMSA=NAVAMSA,
    DASHA=DASHA
)
'''

BaseCONTEXT = getContext(
    YOGS=YOGS,
    CHART=CHART,
)


In [ ]:
len(BaseCONTEXT)

188146

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
n_tokens = len(enc.encode(BaseCONTEXT))
print(f"Context tokens: {n_tokens:,}")

Context tokens: 40,671


## the FirstPass

In [ ]:
# ------------------------------------------------
# FIRST PASS
#
# ------------------------------------------------

FirstPass_Instruction = """
For every Yog or combination found in YOGSs, check if the same Yog is found in CHART.
For every Yog found, give the conditions and the results.
Do no other interpretation
"""

In [ ]:
# ------------------------------------------------
# Assemble the first pass query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(BaseCONTEXT, FirstPass_Instruction)

FirstPass = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)


OPENAI | gpt-6.1-sol | Effort: medium | In: 42,058 | Cached: 0 | Out: 5,301 | Reasoning: 3,353 | Time: 125.0s


In [ ]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(model, FirstPass)
display(Markdown(FirstPass))

Saved: /content/K21_261104_053733_gpt-6.1-sol.md


The matches below use the formation rules in your supplied **“300 Astrology Yogas”** text. Each result is attributed to that text—not presented as a guaranteed outcome. Conditional matches are separated from clearly established ones.

## 1. Established matches

| Yog / combination | Conditions satisfied in CHART | Results stated in the supplied text |
|---|---|---|
| **Sunapha Yoga** | Planets other than the Sun occupy the second from the Moon. The Moon is in Sagittarius; **Jupiter and Saturn occupy Capricorn**, second from it. | Self-earned property and wealth, intelligence, sound decisions, comforts and a respected reputation. |
| **Sakata Yoga — Moon–Jupiter formation** | The Moon occupies the twelfth, sixth or eighth from Jupiter. **Moon in Sagittarius is twelfth from Jupiter in Capricorn.** | Loss and subsequent recovery of fortune, financial fluctuations, hardship, stubbornness and difficulties with relatives. |
| **Vasumathi Yoga** | Benefics occupy an upachaya—3, 6, 10 or 11—from Lagna or Moon. **Venus occupies the third from Cancer Lagna and the tenth from the Sagittarius Moon.** | Wealth through diligence, prosperity, social respect, independence, generous expenditure and helping others. |
| **Amala Yoga** | A benefic occupies the tenth from Moon or Lagna. **Venus in Virgo occupies the tenth from the Sagittarius Moon.** | Lasting fame, a good reputation, upright character, prosperity and health. |
| **Vesi Yoga** | Planets other than Moon occupy the second from Sun. **Mars and Mercury in Libra are second from Sun in Virgo.** | Happiness, good fortune, ethical conduct, fame and an aristocratic standing. |
| **Sasa Yoga** | Saturn occupies a kendra in its own or exaltation sign. **Saturn occupies its own sign Capricorn in the seventh house.** | Authority, command over others and leadership; also possible anger, questionable conduct and desire for others’ wealth. |
| **Parijatha Yoga — first formation alternative** | The relevant dispositor occupies a quadrant, trine or its exaltation sign. The chain is **Lagna lord Moon in Sagittarius → Jupiter in Capricorn → Saturn in Capricorn, in the seventh-house kendra**. | King-like comforts, recognition and respect, particularly in middle and later life; generosity, interest in action and attachment to traditions. |
| **Brahma Yoga** | Jupiter and Venus occupy kendras respectively from the ninth and eleventh lords; Mercury occupies a kendra from Lagna lord or tenth lord. **Jupiter is itself the ninth lord; Venus is itself the eleventh lord; Mercury conjoins tenth lord Mars in Libra.** Each relevant relationship is first-position, hence a kendra. | Luxurious living, learning, respect from learned people, longevity, health and charity. |
| **Kedara Yoga** | The seven non-nodal planets occupy four signs: **Virgo—Sun and Venus; Libra—Mars and Mercury; Sagittarius—Moon; Capricorn—Jupiter and Saturn.** | Livelihood connected with agriculture, helping others, and potentially limited comprehension. |
| **Sareera Soukhya Yoga** | Lagna lord, Jupiter **or** Venus occupies a quadrant. **Jupiter occupies the seventh-house quadrant.** | Bodily comfort, longevity, health, wealth and an interest in politics. |
| **Sada Sanchara Yoga** | Under the supplied wording, Lagna lord or the lord of its occupied sign is placed in a movable sign. **Moon occupies Sagittarius; its sign lord Jupiter occupies movable Capricorn.** | Travel, exploration of unfamiliar places and cultures, meeting new people and a wandering disposition. |
| **Swaveeryaddhana Yoga — second lord in a kendra/trikona from Lagna lord** | **Second lord Sun in Virgo is tenth from Lagna lord Moon in Sagittarius.** | Wealth through personal effort and hardship, self-made prosperity and possible business-building. |
| **Matrumooladdhana Yoga** | Second lord joins or receives the aspect of fourth lord. **Second lord Sun conjoins fourth lord Venus in Virgo.** | Financial assistance or wealth through the mother. |
| **Bhratruvriddhi Yoga** | Third lord, Mars or the third house is joined/aspected by benefics, or is strong. **Venus occupies the third house.** | Happiness through siblings, strong family bonds and prosperity through siblings’ success. |
| **Bandhu Pujya Yoga — Jupiter association alternative** | Fourth house or fourth lord has Jupiter’s association or aspect. **Fourth lord Venus receives Jupiter’s aspect.** | Affection and respect among relatives and associates, empathy and being regarded as trustworthy. |
| **Bandhubhisthyaktha Yoga** | Fourth lord associates with malefics, occupies specified adverse subdivisions, or occupies an inimical/debilitation sign. **Fourth lord Venus is debilitated in Virgo.** | Misunderstanding, disappointing support or desertion by relatives and associates. |
| **Nishkapata Yoga — first formation alternative** | Fourth house contains a benefic, or a planet in exaltation, a friendly sign or its own sign, or is itself a benefic sign. **Mercury occupies the fourth house and is explicitly recorded as being in a friendly house.** | Sincerity, a loving disposition, dislike of hypocrisy and preference for openness rather than secrecy. |
| **Ekaputra Yoga** | Fifth lord occupies a kendra or trikona. **Fifth lord Mars occupies the fourth-house kendra.** | One child, described by the text as most likely a son; limited further progeny. |
| **Satkalatra Yoga** | Seventh lord or Venus joins or receives the aspect of Jupiter or Mercury. **Seventh lord Saturn conjoins Jupiter; Venus also receives Jupiter’s aspect.** | A noble, disciplined, virtuous and supportive spouse. |
| **Raja Yoga — debilitation-relief formation: Jupiter** | The lord of a debilitated planet’s sign, or the planet exalted there, occupies a kendra from Moon or Lagna. **Jupiter is debilitated in Capricorn; Capricorn lord Saturn occupies the seventh from Lagna, while Mars—the planet exalted in Capricorn—occupies the fourth.** | The entry attributes wealth, influence, authority and ruler-like standing to this formation. |
| **Raja Yoga — debilitation-relief formation: Venus** | The same supplied rule applies. **Venus is debilitated in Virgo; Virgo lord Mercury occupies the fourth-house kendra from Lagna.** | The entry attributes wealth, influence, authority and ruler-like standing to this formation. |

## 2. Matches under the supplied text’s literal wording

These formations match the wording provided, but their textual limitations should remain explicit.

| Yog / combination | Conditions satisfied in CHART | Results stated in the supplied text |
|---|---|---|
| **Vasi Yoga** | The rule says planets **other than Moon** occupy the twelfth from Sun. **Rahu in Leo is twelfth from Sun in Virgo.** This matches literally because the supplied rule does not exclude Rahu. | Happiness, prosperity, a liberal disposition and favour from ruling authorities. |
| **Obhayachari Yoga** | Planets other than Moon occupy both sides of Sun. **Rahu occupies Leo, twelfth from Sun; Mars and Mercury occupy Libra, second from Sun.** This likewise depends on including Rahu, which the supplied wording does not exclude. | Eloquence, an impressive presence, happiness, wealth and fame. |
| **Parvata Yoga** | The supplied rule says the **sixth or eighth** should be empty or occupied by benefics. **The sixth contains Moon, explicitly labelled a functional benefic.** The eighth contains Ketu, so this matches the stated **“or”** condition, not a requirement that both houses qualify. | Wealth, prosperity, charity, humour, leadership and passion. |

## 3. Conditional matches using CHART’s functional classifications

The source does not specify whether “benefic” and “malefic” in these particular rules mean **natural** or **functional** status. These therefore remain conditional rather than unqualified matches.

| Yog / combination | Conditions satisfied under that reading | Results stated in the supplied text |
|---|---|---|
| **Daridra Yoga — fifth lord associated with a dusthana lord without benefic association/aspect** | **Fifth lord Mars conjoins twelfth lord Mercury**, which CHART explicitly labels a functional malefic. Mars receives Saturn’s aspect, and no separately qualifying benefic conjunction/aspect is supplied under this functional reading. | Difficulty accumulating wealth, financial distress and hardship. |
| **Kapata Yoga — fourth-house affliction formation** | **Functional malefic Mercury occupies the fourth.** Fourth lord Venus in Virgo is hemmed between **Rahu in Leo and functional malefic Mercury in Libra**. This meets the formation if functional malefic status is accepted for the rule. | A diplomatic manner that may become hypocrisy, inconsistency between stated principles and actions, or a superior attitude toward others. |

**Navamsha-dependent formations and combinations requiring unprovided strength measures, combustion status, birth-time daylight conditions or special subdivisions have not been asserted as established matches.**

In [ ]:
LocatedYogs = FirstPass

## Add Gochar and Transit Info

In [ ]:
Gochar = readText(GOCHAR)
Transits = readText(GOCHAR_EVENTS)

/content/Gochar_Chart.txt | 4,787 characters
/content/Transit.txt | 5,654 characters


In [ ]:
CONTEXT = augContext(
    BaseCONTEXT,
    LOCATEDYOGS=LocatedYogs,
)

## Konsultation

In [ ]:
# ------------------------------------------------
# Ask your question / define the task
# This and subsequent cells can be run multiple times
# ------------------------------------------------
CurrentQuestion = """
For each yoga in LocatedYogs, perform the following audit:
a. **Yoga and Source:** State the yoga name and locate its entry in YOGS. Quote verbatim the exact text defining its formation conditions.
b. **Required Conditions:** Explain clearly, in plain English, the conditions that must be satisfied for this yoga to exist.
c. **Chart Evidence:** Demonstrate how each condition is satisfied (or not satisfied) using explicit planetary positions, house relationships and other relevant information from CHART.
d. **Audit Verdict:** Classify the yoga as VERIFIED, CONDITIONAL, REJECTED or UNVERIFIABLE. Briefly explain the verdict, identifying any missing conditions, ambiguities or contradictions.
Do not assume that a yoga identified in LocatedYogs is necessarily valid. Do not introduce additional yogas or use rules from outside YOGS.
**OUTPUT FORMAT:**
Present each yoga as a separate numbered section, with its name as the heading, followed by paragraphs labelled (a), (b), (c) and (d).
Use clear, readable prose. Do not use tables.
Conclude with a short summary listing the number of VERIFIED, CONDITIONAL, REJECTED and UNVERIFIABLE yogas.
"""

In [ ]:
# ------------------------------------------------
# Assemble the query
# Call the LLM
# ------------------------------------------------

#model = "gpt-6.1-sol"

QUERY = buildQuery(CONTEXT, CurrentQuestion)

answer = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)

OPENAI | gpt-6.1-sol | Effort: medium | In: 44,223 | Cached: 1,335 | Out: 5,814 | Reasoning: 944 | Time: 81.8s


In [ ]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(model, answer)
# ------------------------------------------------
# Convert markdown to docx for human readability
# Second Parameter is identifier of native, can be replaced by id
# ------------------------------------------------
DOCX_FILE = markdownToDocx(MD_FILE, Native_id, CurrentQuestion, provider,model)

Saved: /content/K21_261104_060439_gpt-6.1-sol.md
Saved: /content/K21_261104_060439_gpt-6.1-sol.docx


## Check Credit Status
https://platform.openai.com/settings/organization/billing/credit-grants

https://platform.openai.com/usage


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)